# Sitzung 5: Stammfunktionen & bestimmtes Integral — Interaktiv erkunden

In diesem Notebook kannst du:
1. **Sehen**, was ein Integral geometrisch bedeutet (Fläche unter der Kurve!)
2. **Verstehen**, wie Ableitung und Integral zusammenhängen (Hauptsatz)
3. **Stammfunktionen** berechnen lassen und visuell überprüfen
4. **Bestimmte Integrale** mit farbiger Fläche darstellen
5. **Flächen mit Vorzeichenwechsel** korrekt berechnen

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Text
from IPython.display import display, Markdown
import sympy as sp
%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['text.usetex'] = False
plt.rcParams['mathtext.fontset'] = 'cm'  # Computer Modern (LaTeX-Look)

_x = sp.Symbol('x')

---
## 1. Was ist ein Integral? — Riemannsche Summen

Die Idee: Wir **zerlegen die Fläche** unter einer Kurve in schmale Rechtecke. Je mehr Rechtecke, desto genauer die Annäherung an die tatsächliche Fläche.

> **Überlege zuerst:** Wenn du die Fläche unter f(x) = x² von 0 bis 2 mit nur 2 Rechtecken annäherst — wird die Summe zu groß oder zu klein sein? Warum?

Verschiebe den Slider und überprüfe deine Vermutung!

In [ ]:
def riemann_summen(n_rechtecke=5):
    f = lambda t: t**2  # Funktion
    a, b = 0, 2
    exakt = 8 / 3  # ∫₀² x² dx = 8/3

    xs = np.linspace(a, b, 300)
    dx = (b - a) / n_rechtecke

    fig, ax = plt.subplots()

    # Funktion plotten
    ax.plot(xs, f(xs), 'b-', linewidth=2.5, label=r'$f(x) = x^2$')

    # Rechtecke zeichnen (linke Riemann-Summe)
    summe = 0
    for i in range(n_rechtecke):
        xi = a + i * dx
        hoehe = f(xi)
        rect = plt.Rectangle((xi, 0), dx, hoehe,
                              facecolor='#2196F3', edgecolor='#1565C0',
                              alpha=0.4, linewidth=1)
        ax.add_patch(rect)
        summe += hoehe * dx

    fehler = abs(summe - exakt)
    ax.set_xlim(-0.2, 2.5)
    ax.set_ylim(-0.3, 5)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=13, loc='upper left', framealpha=0.9)
    ax.set_title(
        rf'$n = {n_rechtecke}$ Rechtecke $\;\rightarrow\;$ '
        rf'Summe $= {summe:.4f}$ '
        rf'(exakt: $\frac{{8}}{{3}} \approx {exakt:.4f}$, '
        rf'Fehler: ${fehler:.4f}$)',
        fontsize=13
    )
    plt.tight_layout()
    plt.show()

interact(
    riemann_summen,
    n_rechtecke=IntSlider(value=5, min=1, max=100, step=1,
                          description='Rechtecke:',
                          style={'description_width': '80px'},
                          layout={'width': '450px'})
);

**Beobachte:**
- Mit **wenigen Rechtecken** ist die Annäherung grob — es fehlen Stücke oder es wird zu viel gezählt.
- Mit **vielen Rechtecken** wird die Summe immer genauer.
- Im Grenzfall (n → ∞) ergibt sich das **bestimmte Integral**: ∫₀² x² dx = 8/3.

---
## 2. Hauptsatz anschaulich: f(x) und F(x) nebeneinander

Der **Hauptsatz der Differential- und Integralrechnung** sagt:

> Wenn F eine Stammfunktion von f ist, dann gilt: ∫ₐᵇ f(x) dx = F(b) − F(a)

> **Überlege zuerst:** Wenn f(x) = x² und F(x) = x³/3, was ist dann F(2) − F(0)? Rechne im Kopf!

Hier siehst du f(x) und F(x) gleichzeitig. Die **farbige Fläche** unter f entspricht dem Wert F(b) − F(a). Verschiebe jetzt **beide** Grenzen a und b!

In [ ]:
hauptsatz_funktionen = {
    'x²': (lambda t: t**2, lambda t: t**3 / 3,
            r'$f(x) = x^2$', r'$F(x) = \frac{x^3}{3}$'),
    'sin(x)': (lambda t: np.sin(t), lambda t: -np.cos(t),
               r'$f(x) = \sin(x)$', r'$F(x) = -\cos(x)$'),
    '2x + 1': (lambda t: 2*t + 1, lambda t: t**2 + t,
               r'$f(x) = 2x + 1$', r'$F(x) = x^2 + x$'),
    'eˣ': (lambda t: np.exp(t), lambda t: np.exp(t),
           r'$f(x) = e^x$', r'$F(x) = e^x$'),
}

def hauptsatz_visual(wahl='x²', a_wert=0.0, b_wert=2.0):
    if a_wert >= b_wert:
        print("⚠ a muss kleiner als b sein!")
        return

    f_func, F_func, f_label, F_label = hauptsatz_funktionen[wahl]

    xs = np.linspace(-1, 4, 400)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    # Linkes Bild: f(x) mit Fläche
    ys = f_func(xs)
    ax1.plot(xs, ys, 'b-', linewidth=2.5, label=f_label)

    # Fläche einfärben
    x_fill = np.linspace(a_wert, b_wert, 200)
    y_fill = f_func(x_fill)
    ax1.fill_between(x_fill, 0, y_fill,
                     where=(y_fill >= 0), color='#2196F3', alpha=0.3)
    ax1.fill_between(x_fill, 0, y_fill,
                     where=(y_fill < 0), color='#F44336', alpha=0.3)

    integral_wert = F_func(b_wert) - F_func(a_wert)
    ax1.set_title(
        rf'$\int_{{{a_wert:.1f}}}^{{{b_wert:.1f}}} f(x)\,dx = {integral_wert:.3f}$',
        fontsize=14
    )
    ax1.axhline(y=0, color='k', linewidth=0.5)
    ax1.axvline(x=0, color='k', linewidth=0.5)
    ax1.grid(True, alpha=0.3)
    ax1.legend(fontsize=13, loc='upper left', framealpha=0.9)
    ax1.set_xlim(-1, 4)
    ax1.set_ylim(-2, max(8, np.max(ys[xs <= 4]) * 1.2))

    # Rechtes Bild: F(x) mit Markierungen F(a) und F(b)
    Fs = F_func(xs)
    ax2.plot(xs, Fs, 'g-', linewidth=2.5, label=F_label)
    ax2.plot(a_wert, F_func(a_wert), 'ro', markersize=10, zorder=5)
    ax2.plot(b_wert, F_func(b_wert), 'ro', markersize=10, zorder=5)

    # Labels für F(a) und F(b)
    ax2.annotate(rf'$F(a) = {F_func(a_wert):.3f}$',
                 xy=(a_wert, F_func(a_wert)),
                 xytext=(a_wert - 0.6, F_func(a_wert) + 1),
                 fontsize=11, color='red',
                 arrowprops=dict(arrowstyle='->', color='red'),
                 bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    # Verbindungslinie F(b) - F(a)
    ax2.annotate('', xy=(b_wert + 0.15, F_func(b_wert)),
                 xytext=(b_wert + 0.15, F_func(a_wert)),
                 arrowprops=dict(arrowstyle='<->', color='red', lw=2))
    ax2.text(b_wert + 0.3, (F_func(a_wert) + F_func(b_wert)) / 2,
             rf'$F(b) - F(a) = {integral_wert:.3f}$',
             fontsize=12, color='red', va='center',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax2.set_title(r'Stammfunktion $F(x)$', fontsize=14)
    ax2.axhline(y=0, color='k', linewidth=0.5)
    ax2.axvline(x=0, color='k', linewidth=0.5)
    ax2.grid(True, alpha=0.3)
    ax2.legend(fontsize=13, loc='upper left', framealpha=0.9)
    ax2.set_xlim(-1, 4)

    plt.tight_layout()
    plt.show()

interact(
    hauptsatz_visual,
    wahl=Dropdown(options=list(hauptsatz_funktionen.keys()), value='x²',
                  description='Funktion:'),
    a_wert=FloatSlider(value=0.0, min=-0.5, max=2.5, step=0.1,
                       description='a =',
                       style={'description_width': '30px'},
                       layout={'width': '400px'}),
    b_wert=FloatSlider(value=2.0, min=0.1, max=3.5, step=0.1,
                       description='b =',
                       style={'description_width': '30px'},
                       layout={'width': '400px'}),
);

**Der Hauptsatz in Aktion:**
- Links: Die **blaue Fläche** unter f(x) wächst, wenn du b nach rechts schiebst.
- Rechts: Der **Höhenunterschied** F(b) − F(a) entspricht genau dieser Fläche.
- Probiere verschiedene Funktionen aus!

---
## 3. Stammfunktionen-Rechner

Gib eine beliebige Funktion ein und erhalte die Stammfunktion — symbolisch berechnet mit **sympy** und als Plot visualisiert.

In [ ]:
def _parse_formel(formel_str):
    """Parst eine Formel-Eingabe zu einem sympy-Ausdruck."""
    formel_normiert = formel_str.strip().replace('^', '**')
    return sp.sympify(
        formel_normiert,
        locals={
            'x': _x,
            'e': sp.E,
            'ln': sp.log,
            'log': sp.log,
            'exp': sp.exp,
            'sqrt': sp.sqrt,
            'sin': sp.sin,
            'cos': sp.cos,
            'tan': sp.tan,
        },
    )

def _plot_werte(expr, xs):
    """Berechnet Plot-Werte aus einem sympy-Ausdruck."""
    with np.errstate(all='ignore'):
        werte = sp.lambdify(_x, expr, 'numpy')(xs)
    werte = np.asarray(werte, dtype=np.complex128)
    if werte.ndim == 0:
        werte = np.full(xs.shape, werte, dtype=np.complex128)
    reelle_werte = np.where(np.abs(werte.imag) < 1e-9, werte.real, np.nan)
    return np.where(np.isfinite(reelle_werte), reelle_werte, np.nan)

def stammfunktion_rechner(formel_str):
    """Berechnet Stammfunktion und plottet f(x) und F(x)."""
    try:
        f_sym = _parse_formel(formel_str)
        F_sym = sp.integrate(f_sym, _x)

        display(Markdown(f"**f(x) =** ${sp.latex(f_sym)}$"))
        display(Markdown(f"**F(x) =** ${sp.latex(F_sym)}$ + C"))

        # Gegenprobe
        check = sp.simplify(sp.diff(F_sym, _x) - f_sym)
        if check == 0:
            display(Markdown(r"**Probe:** $F'(x) = f(x)$ &#10003;"))

        xs = np.linspace(-3, 3, 600)
        f_werte = _plot_werte(f_sym, xs)
        F_werte = _plot_werte(F_sym, xs)

        fig, ax = plt.subplots()
        ax.plot(xs, f_werte, 'b-', linewidth=2, label=r'$f(x)$')
        ax.plot(xs, F_werte, 'g--', linewidth=2, label=r'$F(x)$')
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=14, loc='upper left', framealpha=0.9)
        ax.set_xlim(xs[0], xs[-1])
        ax.set_ylim(-10, 10)
        ax.set_title(r'Funktion $f(x)$ und Stammfunktion $F(x)$', fontsize=14)
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Fehler: {e}")

interact(
    stammfunktion_rechner,
    formel_str=Text(
        value='x**2',
        placeholder='z.B. x^3 - 2*x, e^x, 1/x, sin(x)',
        description='f(x) =',
        style={'description_width': '50px'},
        layout={'width': '470px'},
        continuous_update=False,
    ),
);

### Probiere selbst! Ändere den String oben und drücke Enter, z.B.:

| Funktion | Eingabe |
|----------|---------|
| x³ − 2x | `x^3 - 2*x` |
| eˣ | `exp(x)` oder `e^x` |
| 1/x | `1/x` |
| sin(x) | `sin(x)` |
| e^(−2x) | `e^(-2*x)` |
| 3/x² | `3/x^2` |

---
## 4. Bestimmtes Integral berechnen — mit farbiger Fläche

Gib eine Funktion und die Integrationsgrenzen ein. Die Fläche wird farbig dargestellt:
- **Blau** = positive Fläche (Graph oberhalb der x-Achse)
- **Rot** = negative Fläche (Graph unterhalb der x-Achse)

In [ ]:
def bestimmtes_integral_visual(formel_str='x**2 - 1', a=-1.0, b=2.0):
    try:
        f_sym = _parse_formel(formel_str)
        F_sym = sp.integrate(f_sym, _x)
        integral = sp.integrate(f_sym, (_x, sp.Rational(str(a)), sp.Rational(str(b))))

        # Schritt-für-Schritt-Rechnung anzeigen
        a_rat = sp.Rational(str(a))
        b_rat = sp.Rational(str(b))
        F_b = F_sym.subs(_x, b_rat)
        F_a = F_sym.subs(_x, a_rat)

        display(Markdown("**Schritt-für-Schritt:**"))
        display(Markdown(rf"1. Stammfunktion: $F(x) = {sp.latex(F_sym)}$"))
        display(Markdown(rf"2. Obere Grenze einsetzen: $F({sp.latex(b_rat)}) = {sp.latex(F_b)}$"))
        display(Markdown(rf"3. Untere Grenze einsetzen: $F({sp.latex(a_rat)}) = {sp.latex(F_a)}$"))
        display(Markdown(
            rf"4. **Ergebnis:** $F({sp.latex(b_rat)}) - F({sp.latex(a_rat)}) "
            rf"= {sp.latex(F_b)} - ({sp.latex(F_a)}) = {sp.latex(integral)}"
            rf" \approx {float(integral):.4f}$"
        ))

        xs = np.linspace(a - 1, b + 1, 600)
        ys = _plot_werte(f_sym, xs)
        x_fill = np.linspace(a, b, 400)
        y_fill = _plot_werte(f_sym, x_fill)

        fig, ax = plt.subplots()
        ax.plot(xs, ys, 'b-', linewidth=2.5, label=rf'$f(x) = {sp.latex(f_sym)}$')

        # Fläche einfärben (positiv blau, negativ rot)
        ax.fill_between(x_fill, 0, y_fill,
                        where=(y_fill >= 0), color='#2196F3', alpha=0.35,
                        interpolate=True, label='positive Fläche')
        ax.fill_between(x_fill, 0, y_fill,
                        where=(y_fill < 0), color='#F44336', alpha=0.35,
                        interpolate=True, label='negative Fläche')

        # Grenzen markieren
        ax.axvline(x=a, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.axvline(x=b, color='gray', linestyle=':', linewidth=1.5, alpha=0.7)
        ax.axhline(y=0, color='k', linewidth=0.5)
        ax.axvline(x=0, color='k', linewidth=0.5)
        ax.grid(True, alpha=0.3)
        ax.legend(fontsize=12, loc='upper left', framealpha=0.9)
        ax.set_title(
            rf'Bestimmtes Integral: $\int_{{{a:.1f}}}^{{{b:.1f}}} f(x)\,dx = {float(integral):.4f}$',
            fontsize=14
        )
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f"Fehler: {e}")

interact(
    bestimmtes_integral_visual,
    formel_str=Text(
        value='x**2 - 1',
        placeholder='z.B. x^2, sin(x), e^x',
        description='f(x) =',
        style={'description_width': '50px'},
        layout={'width': '400px'},
        continuous_update=False,
    ),
    a=FloatSlider(value=-1.0, min=-5.0, max=4.0, step=0.5,
                  description='a =',
                  style={'description_width': '30px'},
                  layout={'width': '400px'}),
    b=FloatSlider(value=2.0, min=-4.0, max=5.0, step=0.5,
                  description='b =',
                  style={'description_width': '30px'},
                  layout={'width': '400px'}),
);

**Probiere:**
- `x^2 - 1` mit a = −1, b = 1 → Wie viel rote und blaue Fläche gibt es?
- `sin(x)` mit a = 0, b = 2*π → Warum ist das Integral 0?
- `x^3` mit a = −2, b = 2 → Ungerade Funktion!

---
## 5. Fläche mit Vorzeichenwechsel — warum Fläche ≠ Integral

**Das Problem:** Das bestimmte Integral kann **negativ** sein oder sich **aufheben**, wenn der Graph die x-Achse kreuzt. Für den **Flächeninhalt** muss man die Teilflächen getrennt berechnen und die Beträge addieren.

> **Überlege zuerst:** f(x) = x² − 4 hat Nullstellen bei x = −2 und x = 2. Liegt der Graph dazwischen *oberhalb* oder *unterhalb* der x-Achse? Was bedeutet das für ∫₋₂² f(x) dx — positiv oder negativ? Und wie groß ist der **Flächeninhalt**?

In [ ]:
def vorzeichenwechsel_visual(formel_str='x**2 - 4', a=-2.5, b=2.5):
    try:
        f_sym = _parse_formel(formel_str)

        # Nullstellen im Intervall finden
        nullstellen_sym = sp.solve(f_sym, _x)
        nullstellen = sorted([float(n) for n in nullstellen_sym
                              if n.is_real and a < float(n) < b])

        # Plotbereich
        x_min = a - 1
        x_max = b + 1
        xs = np.linspace(x_min, x_max, 600)
        f_np_func = sp.lambdify(_x, f_sym, 'numpy')
        ys = f_np_func(xs)

        # Teilintegrale berechnen
        grenzen = [a] + nullstellen + [b]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # --- Linkes Bild: Integral (mit Vorzeichen) ---
        ax1.plot(xs, ys, 'b-', linewidth=2.5)
        x_fill = np.linspace(a, b, 400)
        y_fill = f_np_func(x_fill)
        ax1.fill_between(x_fill, 0, y_fill,
                         where=(y_fill >= 0), color='#2196F3', alpha=0.35, interpolate=True)
        ax1.fill_between(x_fill, 0, y_fill,
                         where=(y_fill < 0), color='#F44336', alpha=0.35, interpolate=True)

        integral_gesamt = float(sp.integrate(f_sym, (_x, a, b)))
        ax1.set_title(
            rf'Integral: $\int_{{{a:.1f}}}^{{{b:.1f}}} ({sp.latex(f_sym)})\,dx = {integral_gesamt:.3f}$',
            fontsize=13
        )
        ax1.axhline(y=0, color='k', linewidth=0.5)
        ax1.grid(True, alpha=0.3)
        ax1.set_xlabel('Positive und negative Anteile heben sich auf!', fontsize=11,
                       color='gray')

        # --- Rechtes Bild: Flächeninhalt (Beträge) ---
        ax2.plot(xs, ys, 'b-', linewidth=2.5)
        flaeche_gesamt = 0
        farben = ['#2196F3', '#F44336', '#4CAF50', '#FF9800', '#9C27B0']
        for i in range(len(grenzen) - 1):
            gi, gf = grenzen[i], grenzen[i + 1]
            teil_integral = float(sp.integrate(f_sym, (_x, gi, gf)))
            teil_flaeche = abs(teil_integral)
            flaeche_gesamt += teil_flaeche

            x_teil = np.linspace(gi, gf, 200)
            y_teil = f_np_func(x_teil)
            ax2.fill_between(x_teil, 0, y_teil, alpha=0.4,
                             color=farben[i % len(farben)])
            # Betrag annotieren
            x_mitte = (gi + gf) / 2
            y_mitte = f_np_func(x_mitte) / 2
            ax2.text(x_mitte, y_mitte, rf'$|{teil_integral:.2f}|$',
                     fontsize=11, ha='center', va='center',
                     bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

        ax2.set_title(
            rf'Flächeninhalt: $A = {flaeche_gesamt:.3f}$',
            fontsize=13
        )
        ax2.axhline(y=0, color='k', linewidth=0.5)
        ax2.grid(True, alpha=0.3)
        ax2.set_xlabel('Beträge der Teilflächen werden addiert!', fontsize=11,
                       color='gray')

        for ax in (ax1, ax2):
            ax.set_xlim(x_min, x_max)

        plt.tight_layout()
        plt.show()

        # Zusammenfassung mit Schritten
        if nullstellen:
            ns_str = ", ".join([f"{n:.2f}" for n in nullstellen])
            display(Markdown(f"**Nullstellen im Intervall:** x = {ns_str}"))

        if abs(integral_gesamt - flaeche_gesamt) > 0.001:
            display(Markdown(
                rf"**Integral** = {integral_gesamt:.3f} &ne; "
                rf"**Flächeninhalt** = {flaeche_gesamt:.3f} "
                r"&rarr; Teile das Intervall an den Nullstellen auf!"
            ))
        else:
            display(Markdown(
                rf"**Integral** = **Flächeninhalt** = {flaeche_gesamt:.3f} "
                r"(Graph liegt komplett auf einer Seite der x-Achse)"
            ))
    except Exception as e:
        print(f"Fehler: {e}")

interact(
    vorzeichenwechsel_visual,
    formel_str=Text(
        value='x**2 - 4',
        placeholder='z.B. x^2 - 4, x^3 - 4*x, sin(x)',
        description='f(x) =',
        style={'description_width': '50px'},
        layout={'width': '400px'},
        continuous_update=False,
    ),
    a=FloatSlider(value=-2.5, min=-5.0, max=4.0, step=0.5,
                  description='a =',
                  style={'description_width': '30px'},
                  layout={'width': '400px'}),
    b=FloatSlider(value=2.5, min=-4.0, max=5.0, step=0.5,
                  description='b =',
                  style={'description_width': '30px'},
                  layout={'width': '400px'}),
);

**Merke:**
- **Integral** = vorzeichenbehaftete Fläche (kann 0 oder negativ sein!)
- **Flächeninhalt** = Summe der Beträge der Teilflächen
- **Vorgehen im Abitur:** Nullstellen bestimmen → Intervall aufteilen → Teilintegrale berechnen → Beträge addieren

---
## 6. Integralregeln live testen

Analog zum Ableitungsrechner aus Sitzung 1: Gib eine beliebige Funktion ein und erhalte das unbestimmte Integral, das bestimmte Integral und die Probe durch Ableiten.

In [ ]:
integralregeln = {
    'Potenzregel': [
        ('x^3', r'\int x^3\,dx'),
        ('x^5 - 3*x^2 + 7', r'\int (x^5 - 3x^2 + 7)\,dx'),
        ('1/x^3', r'\int \frac{1}{x^3}\,dx'),
        ('sqrt(x)', r'\int \sqrt{x}\,dx'),
    ],
    'e-Funktion': [
        ('exp(x)', r'\int e^x\,dx'),
        ('exp(2*x)', r'\int e^{2x}\,dx'),
        ('exp(-3*x)', r'\int e^{-3x}\,dx'),
        ('x * exp(x)', r'\int x \cdot e^x\,dx'),
    ],
    'Logarithmus & 1/x': [
        ('1/x', r'\int \frac{1}{x}\,dx'),
        ('1/(2*x)', r'\int \frac{1}{2x}\,dx'),
        ('3/x', r'\int \frac{3}{x}\,dx'),
    ],
    'Trigonometrisch': [
        ('sin(x)', r'\int \sin(x)\,dx'),
        ('cos(x)', r'\int \cos(x)\,dx'),
        ('sin(2*x)', r'\int \sin(2x)\,dx'),
    ],
}

def integralregeln_explorer(kategorie='Potenzregel'):
    beispiele = integralregeln[kategorie]

    display(Markdown(f"### Kategorie: {kategorie}"))
    display(Markdown("---"))

    for formel_str, integral_latex in beispiele:
        f_sym = _parse_formel(formel_str)
        F_sym = sp.integrate(f_sym, _x)
        probe = sp.simplify(sp.diff(F_sym, _x) - f_sym)
        probe_ok = "&#10003;" if probe == 0 else "&#10007;"

        display(Markdown(
            rf"${integral_latex} = {sp.latex(F_sym)} + C$ "
            rf"&nbsp;&nbsp; Probe: $F'(x) = {sp.latex(sp.diff(F_sym, _x))}$ {probe_ok}"
        ))

    display(Markdown("---"))
    display(Markdown(
        "**Tipp:** Im Abschnitt 3 (Stammfunktionen-Rechner) kannst du "
        "eigene Funktionen eingeben und testen!"
    ))

interact(
    integralregeln_explorer,
    kategorie=Dropdown(
        options=list(integralregeln.keys()),
        value='Potenzregel',
        description='Kategorie:',
        style={'description_width': '80px'},
    ),
);

---
## Zusammenfassung

| Regel | Funktion f(x) | Stammfunktion F(x) |
|-------|---------------|-------------------|
| Potenzregel | x^n (n ≠ −1) | x^(n+1) / (n+1) + C |
| Exponential | e^(ax) | e^(ax) / a + C |
| Logarithmus | 1/x | ln\|x\| + C |
| Sinus | sin(ax) | −cos(ax) / a + C |
| Kosinus | cos(ax) | sin(ax) / a + C |

**Hauptsatz:** ∫ₐᵇ f(x) dx = F(b) − F(a)

**Flächeninhalt bei Vorzeichenwechsel:** Nullstellen bestimmen, Teilintegrale berechnen, Beträge addieren.

---
### Desmos zum Weiter-Erkunden

Auf [desmos.com/calculator](https://www.desmos.com/calculator) kannst du Integrale auch visualisieren:
- Funktion eingeben, z.B. `f(x) = x^2 - 4`
- Integral anzeigen: `\int_{-2}^{2} f(t) dt` (Desmos berechnet das automatisch!)